# ViFinQA -- Kaggle GPU: write the synthetic questions, then measure X on them

`EXEC = coverage x P(gold table in prompt) x X`. The first two are measured off the scoreboard.
`X` -- the chance the model writes a correct program when it has a program to write and the gold
table in front of it -- has only ever been divided out of the other two, and every change aimed at
it has cost a twelve-hour run plus a submission to evaluate.

`outputs/synthetic/dev_programs.jsonl` holds 499 programs over real tables, executed, so their
answers are right by construction. What it lacks is the Vietnamese question. This notebook adds
it and then puts each question to the shipped solver prompt:

1. `71_render_questions.py` writes the question. The model never sees a value and never computes,
   so a bad generation costs one rejected sample and can never corrupt an answer.
2. `73_filter_synthetic.py` hands question and gold tables to `build_program_prompt` -- the
   production prompt itself, not a copy -- and records how many independent tries came back with
   the right number.

**What the resulting number is, said precisely.** `73` shows the solver the *gold* tables only,
while production shows it about twenty candidates. So `solved / attempts` here is X with the
retrieval difficulty removed: an **upper bound** on the production X, and the right quantity for
comparing two prompts against each other. Pass `--distractors 19` to measure at production width
instead, at the cost of a much longer prefill per question.

Once this set exists, every later question about the prompt -- few-shot examples, row hierarchy,
two-stage table selection -- is answered here in minutes instead of by a submission.

In [ ]:
# This notebook needs no submission, no final-run gate and no dense index. It needs a served
# model and the synthetic programs, and it produces the first measurement of X in the project.
import os

MODEL_PROFILES = {
    "qwen3_8b_awq": {
        "model": "Qwen/Qwen3-8B-AWQ",
        "revision": "4da05a8edb55c6046cce958586c33b61da07bb79",
        "total_params_b": "8.2",
        "non_embedding_params_b": "6.95",
        "max_num_seqs": "4",
    },
    "qwen3_14b_awq": {
        "model": "Qwen/Qwen3-14B-AWQ",
        "revision": "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
        "total_params_b": "14.8",
        "non_embedding_params_b": "13.2",
        "max_num_seqs": "2",
    },
}
# Measure X on the model that will answer the paper. A different model here measures a different
# quantity, and the comparison it was built for stops being one.
os.environ["VIFINQA_MODEL_PROFILE"] = "qwen3_8b_awq"
profile = MODEL_PROFILES[os.environ["VIFINQA_MODEL_PROFILE"]]
os.environ["VIFINQA_MODEL"] = profile["model"]
os.environ["VIFINQA_MODEL_REVISION"] = profile["revision"]
os.environ["VIFINQA_MODEL_TOTAL_PARAMS_B"] = profile["total_params_b"]
os.environ["VIFINQA_MODEL_NON_EMBEDDING_PARAMS_B"] = profile["non_embedding_params_b"]
os.environ["VIFINQA_MAX_NUM_SEQS"] = profile["max_num_seqs"]
os.environ["VIFINQA_THINKING_MODE"] = "disabled"
os.environ["VIFINQA_TABLE_UNIT_SOURCE"] = "latest"
os.environ["VIFINQA_MAX_TOKENS"] = "6144"
os.environ["VIFINQA_TP"] = "1"
os.environ["VIFINQA_DP"] = "2"
# Nothing here is a submission, so a pinned SHA is a convenience rather than a gate. Leave it
# empty and the run records whatever ref it checked out.
os.environ["VIFINQA_EXPECTED_PROJECT_SHA"] = ""
os.environ["VIFINQA_GIT_REF"] = "main"
os.environ["VIFINQA_ORGANIZER_CONFIRMED_14B"] = "1"
os.environ["VIFINQA_DENSE_REVISION"] = "5617a9f61b028005a4858fdac845db406aefb181"
os.environ["VIFINQA_RERANKER_REVISION"] = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
# Borrowed cells below read these; nothing in this notebook runs a final or subset generation.
os.environ["VIFINQA_FINAL_RUN"] = "0"
os.environ["VIFINQA_RUN_FULL"] = "0"
os.environ["VIFINQA_RUN_SUBSET_200"] = "0"

# Which side of the ticker split to work on. `dev` is the 499-sample measuring set; `train` is the
# 796-sample pool few-shot examples and any fine-tuning would come from. They share no ticker.
SPLIT = "dev"
# Independent solver tries per question. Two is the point where a miss means the question resists
# more than one reading rather than one unlucky decode, and it keeps 499 questions inside one
# session. Three costs half again as long.
SOLVER_ATTEMPTS = 2
# Distractor tables padded into the solver prompt. 0 measures X with retrieval removed, which is
# the upper bound and the cheap comparison; 19 measures it at the width production actually shows.
SOLVER_DISTRACTORS = 0
# Stop after this many questions. Empty means all of them.
SAMPLE_LIMIT = ""
print("split/attempts/distractors/limit:", SPLIT, SOLVER_ATTEMPTS, SOLVER_DISTRACTORS,
      SAMPLE_LIMIT or "all")

In [ ]:
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch

def iter_input_paths(
    relative: str, root: Path = Path("/kaggle/input"), max_depth: int = 12
) -> list[Path]:
    """Return every existing `<directory>/relative` under `root`, following symlinked mounts."""
    matches: list[Path] = []
    visited: set[str] = set()
    for parent, directories, _ in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        if len(Path(parent).parts) - len(root.parts) >= max_depth:
            directories.clear()
        candidate = Path(parent) / relative
        if candidate.exists():
            matches.append(candidate)
    return sorted(matches, key=str)


def describe_inputs(root: Path = Path("/kaggle/input"), max_depth: int = 5) -> str:
    """Return a compact inventory of mounted inputs so failures name what is actually attached.

    Kaggle spends three levels on `datasets/<owner>/<slug>` before any content, so the
    default depth has to reach past the mount point itself.
    """
    if not root.is_dir():
        return f"{root} does not exist"
    lines: list[str] = []
    visited: set[str] = set()
    for parent, directories, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        depth = len(Path(parent).parts) - len(root.parts)
        entries = sorted(filenames)[:4]
        if len(filenames) > 4:
            entries.append(f"+{len(filenames) - 4} more files")
        if depth >= max_depth:
            if directories:
                entries.append(f"+{len(directories)} more directories")
            directories.clear()
        directories.sort()
        lines.append(f"{'  ' * depth}{Path(parent).name or root}/ {entries}")
        if len(lines) >= 80:
            lines.append("... truncated")
            break
    return "\n".join(lines)

try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    del hf_token
    print("authenticated Hugging Face downloads enabled")

print(
    "torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count()
)
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing."
requested_dp = int(os.environ.get("VIFINQA_DP", "2"))
assert (
    torch.cuda.device_count() >= requested_dp
), f"VIFINQA_DP={requested_dp} requires at least {requested_dp} GPUs; select T4 x2."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    capability = torch.cuda.get_device_capability(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), "GiB", "sm", capability)
    assert capability >= (
        7,
        5,
    ), f"GPU {i} {p.name} has capability {capability}; select T4 x2 (sm_75 or newer)."
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
RUN_SUBSET_200 = os.environ.get("VIFINQA_RUN_SUBSET_200") == "1"
assert sum((RUN_FULL, RUN_SUBSET_200)) <= 1, "Full and subset modes are mutually exclusive."
assert not FINAL_RUN or RUN_FULL, "FINAL_RUN=1 is only valid together with RUN_FULL=1."
MODEL_PROFILE = os.environ["VIFINQA_MODEL_PROFILE"]
MODEL_RUN_TAG = MODEL_PROFILE.replace("-", "_")
MODEL = os.environ["VIFINQA_MODEL"]
MODEL_REVISION = os.environ.get("VIFINQA_MODEL_REVISION")
MODEL_TOTAL_PARAMS_B = float(os.environ["VIFINQA_MODEL_TOTAL_PARAMS_B"])
MODEL_NON_EMBEDDING_PARAMS_B = float(
    os.environ["VIFINQA_MODEL_NON_EMBEDDING_PARAMS_B"]
)
DENSE_REVISION = os.environ.get("VIFINQA_DENSE_REVISION")
RERANKER = "BAAI/bge-reranker-v2-m3"
RERANKER_REVISION = os.environ.get("VIFINQA_RERANKER_REVISION")
THINKING_MODE = os.environ.get("VIFINQA_THINKING_MODE", "disabled")
TABLE_UNIT_SOURCE = os.environ.get("VIFINQA_TABLE_UNIT_SOURCE", "latest")
MAX_TOKENS = os.environ.get("VIFINQA_MAX_TOKENS", "4096")
assert MAX_TOKENS.isdigit() and int(MAX_TOKENS) > 0, "VIFINQA_MAX_TOKENS must be a positive integer."
EXPECTED_PROJECT_SHA = os.environ.get("VIFINQA_EXPECTED_PROJECT_SHA", "").strip()
ORGANIZER_CONFIRMED_14B = os.environ.get("VIFINQA_ORGANIZER_CONFIRMED_14B") == "1"
assert THINKING_MODE in {"disabled", "auto"}
assert TABLE_UNIT_SOURCE in {"manifest", "latest"}
if EXPECTED_PROJECT_SHA:
    assert len(EXPECTED_PROJECT_SHA) == 40 and all(
        character in "0123456789abcdef" for character in EXPECTED_PROJECT_SHA
    ), "VIFINQA_EXPECTED_PROJECT_SHA must be a full lowercase Git SHA."
if FINAL_RUN:
    if MODEL_TOTAL_PARAMS_B > 14:
        assert ORGANIZER_CONFIRMED_14B, (
            "Qwen3-14B reports 14.8B total parameters. Use the 8B profile or set "
            "VIFINQA_ORGANIZER_CONFIRMED_14B=1 only with written organiser approval."
        )
    for name, revision in {
        "model": MODEL_REVISION,
        "dense": DENSE_REVISION,
        "reranker": RERANKER_REVISION,
    }.items():
        valid_revision = (
            revision and len(revision) == 40 and all(c in "0123456789abcdef" for c in revision)
        )
        assert valid_revision, f"Final run requires a full lowercase commit SHA for {name}."

In [ ]:
# Prefer attached code. Otherwise clone anonymously, then use a Kaggle GITHUB_TOKEN secret.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
GIT_REF = os.environ.get("VIFINQA_GIT_REF", "main").strip()
assert GIT_REF and not GIT_REF.startswith("-"), "Invalid VIFINQA_GIT_REF."
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
assert PROJECT.parent == Path("/kaggle/working")


def remove_partial_checkout() -> None:
    if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
        shutil.rmtree(PROJECT)


def clone_repo() -> None:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(PROJECT)],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return
    remove_partial_checkout()
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        detail = (result.stderr or "git clone failed").strip().splitlines()[-1]
        raise RuntimeError(
            f"{detail} Enable Internet and either make the repo public, attach the code as a "
            "Kaggle Dataset, or add a read-only GITHUB_TOKEN under Add-ons > Secrets."
        )
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        }
    )
    authenticated = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(PROJECT)],
        env=git_env,
        capture_output=True,
        text=True,
    )
    del token, auth, git_env
    if authenticated.returncode != 0:
        remove_partial_checkout()
        raise RuntimeError("Authenticated clone failed; verify the read-only GITHUB_TOKEN.")


remove_partial_checkout()
attached_candidates = sorted(
    {
        marker.parent
        for marker in iter_input_paths("pyproject.toml")
        if (marker.parent / "scripts/50_generate_programs.py").is_file()
    },
    key=str,
)
if not PROJECT.exists():
    if attached_candidates:
        shutil.copytree(attached_candidates[0], PROJECT)
    else:
        clone_repo()
if (PROJECT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", GIT_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
assert (PROJECT / "pyproject.toml").exists(), f"Invalid project checkout: {PROJECT}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-gpu.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
subprocess.run(
    [
        sys.executable,
        "-c",
        "import bm25s, faiss, ftfy, lxml, openai, pandas, pyarrow, rapidfuzz; "
        "import sentence_transformers, Stemmer, unidecode",
    ],
    check=True,
)
package_probe = (
    "import importlib.metadata as m; import torch; "
    "print('resolved runtime:', 'torch', torch.__version__, 'vllm', m.version('vllm'), "
    "'sentence-transformers', m.version('sentence-transformers'), "
    "'transformers', m.version('transformers'), 'openai', m.version('openai')); "
    "assert torch.__version__.startswith('2.10.'); "
    "assert m.version('vllm') == '0.19.1'; "
    "assert m.version('sentence-transformers') == '5.5.1'; "
    "assert m.version('transformers') == '5.5.3'; "
    "assert m.version('openai').split('.')[0] == '2'"
)
subprocess.run([sys.executable, "-c", package_probe], check=True)
os.chdir(PROJECT)
PROJECT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "attached-archive-no-git-sha"
)
if EXPECTED_PROJECT_SHA:
    assert PROJECT_SHA == EXPECTED_PROJECT_SHA, (
        f"Kaggle checked out {PROJECT_SHA}, expected {EXPECTED_PROJECT_SHA}. "
        "Stop: this is not the published experiment snapshot."
    )
if RUN_SUBSET_200 or FINAL_RUN:
    assert len(PROJECT_SHA) == 40, "Subset/final runs require a Git checkout with a SHA."
    if not EXPECTED_PROJECT_SHA:
        print(
            f"no SHA was declared; this run is pinned to {PROJECT_SHA} from ref {GIT_REF}. "
            "Record it with the results."
        )
print("project revision:", PROJECT_SHA)
RUNTIME_LOG = Path("/kaggle/working/runtime_environment.txt")
RUNTIME_LOG.write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)

In [ ]:
# Discover inputs by their contents, because Kaggle dataset slugs and nesting can vary.
INPUT_INVENTORY = describe_inputs()
print("Kaggle inputs:\n" + INPUT_INVENTORY)
data_candidates = sorted(
    {
        questions.parent.parent
        for questions in iter_input_paths("questions/questions.jsonl")
        if (questions.parent.parent / "code_stock.csv").is_file()
        and (questions.parent.parent / "financial_statements").is_dir()
    },
    key=str,
)
assert data_candidates, (
    "ViFinQA files are not mounted. Attach the `vifinqa` input.\n" f"{INPUT_INVENTORY}"
)
DATA_ROOT = data_candidates[0]
manifest_candidates = sorted(
    {
        manifest
        for manifest in iter_input_paths("processed/table_manifest.jsonl")
        if manifest.with_suffix(".parquet").is_file()
    },
    key=str,
)
assert manifest_candidates, (
    "Frozen artifacts are not mounted. Attach the `vifinqa-artifacts` input.\n"
    f"{INPUT_INVENTORY}"
)
MANIFEST = manifest_candidates[0].with_suffix(".parquet")


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# The tables a rendered question is judged against have to be the ones it was sampled from.
assert (
    sha256(MANIFEST) == "060bd26eff14d30ce70b3ba7b00af509be6100b58ddb5a0fe970afa0ef69e29d"
), "The manifest differs from the one the synthetic programs were sampled against."

# `outputs/` is outside git by design, so the programs arrive as a Dataset rather than with the
# code. Upload `outputs/synthetic/` and attach it.
PROGRAM_FILE = "dev_programs.jsonl" if SPLIT == "dev" else "train_pool.jsonl"
program_candidates = sorted(
    {
        *iter_input_paths(PROGRAM_FILE),
        *iter_input_paths(f"synthetic/{PROGRAM_FILE}"),
    },
    key=str,
)
assert program_candidates, (
    f"No {SPLIT} programs mounted. Upload `outputs/synthetic/` as a Kaggle Dataset and attach "
    f"it; this notebook cannot sample them, because 70_sample_programs.py needs no GPU and "
    f"belongs on a laptop.\n{INPUT_INVENTORY}"
)
PROGRAMS = program_candidates[0]
program_rows = [
    json.loads(line) for line in PROGRAMS.read_text(encoding="utf-8").splitlines() if line.strip()
]
families = sorted({str(row.get("family")) for row in program_rows})
# The Hard family is the one whose question cannot be written from its row label alone: a second
# line item decides which years count. Samples drawn before that was recorded carry no `condition`
# and would be rendered as plain extremums, whose answers the sampler proved differ -- so they
# would be written fluently, fail the solver, and quietly take the 21.3% tier out of the set.
hard = [row for row in program_rows if str(row.get("family")) == "conditional"]
unphrasable = [row for row in hard if not row.get("condition")]
# Recover the field, do not re-sample: these sets were drawn on an earlier revision and the
# current sampler returns 416 samples with 23 Hard ones against the 106 on disk, so regenerating
# would trade a set that mirrors the paper's difficulty mix for one that does not.
assert not unphrasable, (
    f"{len(unphrasable)} of {len(hard)} Hard samples carry no `condition`. Patch the file in "
    "place on a laptop -- `python scripts/74_backfill_condition.py "
    f"outputs/synthetic/{PROGRAM_FILE}` (CPU, no GPU, no model) -- and upload it again. Do NOT "
    "re-run 70_sample_programs.py: it no longer reproduces these sets."
)
ARTIFACTS = Path("/kaggle/working/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
RENDERED = ARTIFACTS / f"{SPLIT}_questions.jsonl"
SOLVED = ARTIFACTS / f"{SPLIT}_solved.jsonl"
REJECTED = ARTIFACTS / f"{SPLIT}_rejected.jsonl"
# Both scripts append and skip what they already judged, so a session that runs out of time is
# resumed by attaching its output here and running again.
for finished in (RENDERED, SOLVED, REJECTED):
    prior = iter_input_paths(finished.name)
    if prior and not finished.exists():
        shutil.copy2(prior[0], finished)
        finished.chmod(0o644)
        print("resuming from", prior[0])
print("programs:", PROGRAMS, len(program_rows), "families:", families)

In [ ]:
# Replicate the model once per T4; full generation sends two independent shards.
#
# Qwen3-14B-AWQ is the tighter fit. Weights land near 8.6 GiB of the 13.8 GiB that
# --gpu-memory-utilization 0.90 leaves on a T4, and its KV cache costs 160 KiB per token
# (40 layers x 8 KV heads x 128 head_dim x 2 x fp16). That leaves room for roughly 34k
# cached tokens, so --max-model-len 16384 fits about two sequences at once rather than the
# four Qwen3-8B-AWQ afforded. MAX_NUM_SEQS defaults down accordingly; raise it only after a
# smoke run shows headroom, because over-subscribing makes vLLM preempt and recompute.
TP = int(os.environ.get("VIFINQA_TP", "1"))
MAX_NUM_SEQS = int(os.environ.get("VIFINQA_MAX_NUM_SEQS", "2"))
DP = int(os.environ.get("VIFINQA_DP", "2"))
assert TP in {1, 2}, "Tensor parallelism above 2 has no second pair of T4s to use."
assert 1 <= DP <= torch.cuda.device_count(), "VIFINQA_DP must not exceed the GPU count."
# Two client shards per replica keep a batch forming while one shard validates and
# executes the program it just received.
SHARDS = DP * int(os.environ.get("VIFINQA_SHARDS_PER_REPLICA", "2"))
# The generator sizes each question's token budget against this, so the server and
# the client must read the same number rather than two copies of it.
MAX_MODEL_LEN = 16384
VLLM_BASE = "http://127.0.0.1:8000"
VLLM_CONFIG = Path("/kaggle/working/vllm_server_config.json")
expected_server_config = {
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "tensor_parallel_size": TP,
    "data_parallel_size": DP,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_model_len": MAX_MODEL_LEN,
    "quantization": "awq_marlin",
}


def cuda_driver_linker_environment() -> dict[str, str]:
    # Kaggle exposes libcuda.so.1 at runtime but its CUDA image can omit the
    # unversioned libcuda.so linker name needed by FlashInfer JIT.
    candidates = []
    ldconfig = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, check=False)
    for line in ldconfig.stdout.splitlines():
        if "libcuda.so.1" in line and "=>" in line:
            candidates.append(Path(line.rsplit("=>", 1)[1].strip()))
    candidates.extend(
        [
            Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
            Path("/usr/local/nvidia/lib64/libcuda.so.1"),
            Path("/usr/lib64/libcuda.so.1"),
        ]
    )
    driver_library = next((path for path in candidates if path.is_file()), None)
    if driver_library is None:
        raise RuntimeError("CUDA driver is active but libcuda.so.1 was not found.")

    linker_dir = Path("/kaggle/working/cuda-driver-link")
    linker_dir.mkdir(parents=True, exist_ok=True)
    linker_name = linker_dir / "libcuda.so"
    if linker_name.is_symlink() or linker_name.exists():
        linker_name.unlink()
    linker_name.symlink_to(driver_library.resolve())

    environment = os.environ.copy()
    for variable in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        existing = [item for item in environment.get(variable, "").split(os.pathsep) if item]
        environment[variable] = os.pathsep.join(
            [str(linker_dir), *[item for item in existing if item != str(linker_dir)]]
        )

    probe_path = linker_dir / "cuda_link_probe"
    probe = subprocess.run(
        ["c++", "-x", "c++", "-", f"-L{linker_dir}", "-lcuda", "-o", str(probe_path)],
        input="int main() { return 0; }\n",
        capture_output=True,
        text=True,
        env=environment,
    )
    probe_path.unlink(missing_ok=True)
    if probe.returncode:
        raise RuntimeError(f"CUDA driver linker probe failed:\n{probe.stderr}")
    print("verified CUDA driver linker:", linker_name, "->", driver_library)
    return environment


def served_model_ids() -> set[str]:
    try:
        health = requests.get(f"{VLLM_BASE}/health", timeout=2)
        if not health.ok:
            return set()
        response = requests.get(f"{VLLM_BASE}/v1/models", timeout=5)
        response.raise_for_status()
        return {item["id"] for item in response.json().get("data", [])}
    except (requests.RequestException, KeyError, TypeError, ValueError):
        return set()


existing_models = served_model_ids()
if existing_models:
    assert (
        MODEL in existing_models
    ), f"Port 8000 already serves {sorted(existing_models)}, not {MODEL}. Restart the session."
    assert (
        VLLM_CONFIG.exists()
    ), "A pre-existing vLLM server has unknown TP/DP; restart the session."
    actual_server_config = json.loads(VLLM_CONFIG.read_text(encoding="utf-8"))
    assert (
        actual_server_config == expected_server_config
    ), f"Existing vLLM config {actual_server_config} != {expected_server_config}; restart."
    print("reusing healthy vLLM server:", sorted(existing_models))
else:
    server_environment = cuda_driver_linker_environment()
    server_log = open("/kaggle/working/vllm.log", "a", encoding="utf-8")  # noqa: SIM115
    serve_cmd = [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--tensor-parallel-size",
        str(TP),
        "--data-parallel-size",
        str(DP),
        "--api-server-count",
        "1",
        "--dtype",
        "half",
        "--quantization",
        # vLLM prints "Detected that the model can run with awq_marlin, however you
        # specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin
        # for faster inference" -- the fast kernel was available on this T4 all along and the
        # explicit flag was turning it off. Decode ran at 3.33 tok/s, which is what made a
        # 2048-token budget unreachable inside a 360 s request timeout.
        "awq_marlin",
        "--max-model-len",
        "16384",
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--gpu-memory-utilization",
        "0.90",
        "--generation-config",
        "vllm",
        "--default-chat-template-kwargs",
        '{"enable_thinking": false}',
        "--seed",
        "20260802",
    ]
    if MODEL_REVISION:
        serve_cmd += ["--revision", MODEL_REVISION]
    server = subprocess.Popen(
        serve_cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_environment
    )

    for _ in range(120):
        if MODEL in served_model_ids():
            break
        if server.poll() is not None:
            log_tail = Path("/kaggle/working/vllm.log").read_text(
                encoding="utf-8", errors="replace"
            )[-50_000:]
            raise RuntimeError(log_tail)
        time.sleep(5)
    else:
        raise TimeoutError("vLLM did not become healthy; inspect /kaggle/working/vllm.log")
    VLLM_CONFIG.write_text(json.dumps(expected_server_config, indent=2) + "\n", encoding="utf-8")
    print("vLLM ready:", MODEL, MODEL_REVISION, "tp/dp=", TP, DP)

In [ ]:
# Write the Vietnamese question for each program. The model is shown the company, the period, the
# line item, the unit and -- for the Hard family -- the filter that decides which years count. It
# is never shown a value and never asked to compute, so nothing it returns can move an answer.
render_cmd = [
    sys.executable,
    "scripts/71_render_questions.py",
    str(PROGRAMS),
    "--output",
    str(RENDERED),
    "--companies",
    str(DATA_ROOT / "code_stock.csv"),
    "--model",
    MODEL,
    "--base-url",
    f"{VLLM_BASE}/v1",
    "--max-tokens",
    "256",
    "--attempts",
    "2",
]
if SAMPLE_LIMIT:
    render_cmd += ["--limit", str(SAMPLE_LIMIT)]
render_started = time.monotonic()
subprocess.run(render_cmd, check=True)
rendered_rows = [
    json.loads(line) for line in RENDERED.read_text(encoding="utf-8").splitlines() if line.strip()
]
render_minutes = (time.monotonic() - render_started) / 60
print(f"rendered {len(rendered_rows)}/{len(program_rows)} in {render_minutes:.1f} min")
# A family that loses most of its samples here is a phrasing bug, not a hard tier: the renderer
# rejects a question that copied the row label or came back without a question mark, and both
# failures cluster by family.
for family in families:
    drawn = sum(1 for row in program_rows if str(row.get("family")) == family)
    written = sum(1 for row in rendered_rows if str(row.get("family")) == family)
    print(f"  {family:12s} {written:4d}/{drawn:4d}")
# The filter costs 8-9 hours. Spending them on a set the renderer mangled is the expensive
# mistake available here, and the renderer's own rejection counts say which failure it was:
# `copied_row_label` means the paraphrase instruction is not landing, `no_question_mark` means
# the schema is, `conditional_without_condition` means the upload is the unpatched file.
RENDER_FLOOR = float(os.environ.get("VIFINQA_RENDER_FLOOR", "0.85"))
rendered_share = len(rendered_rows) / len(program_rows)
assert rendered_share >= RENDER_FLOOR, (
    f"Only {rendered_share:.1%} of {len(program_rows)} programs got a question, under the "
    f"{RENDER_FLOOR:.0%} floor. Read the rejection counts printed above: they name the cause. "
    "Re-running appends and skips what is already written, so a fix costs only the missing "
    "questions. To proceed anyway -- say, on a deliberate --limit run -- set "
    "VIFINQA_RENDER_FLOOR=0 in the config cell and re-run this one."
)

In [ ]:
# Put every rendered question to the production solver prompt and count the tries that came back
# with the right number. This is the measurement: `73` imports `build_program_prompt` rather than
# copying it, so a question it certifies is one the shipped pipeline can answer.
filter_cmd = [
    sys.executable,
    "scripts/73_filter_synthetic.py",
    str(RENDERED),
    "--output",
    str(SOLVED),
    "--rejected",
    str(REJECTED),
    "--manifest",
    str(MANIFEST),
    "--data-root",
    str(DATA_ROOT),
    "--model",
    MODEL,
    "--base-url",
    f"{VLLM_BASE}/v1",
    "--table-unit-source",
    TABLE_UNIT_SOURCE,
    "--thinking-mode",
    THINKING_MODE,
    "--max-tokens",
    MAX_TOKENS,
    "--context-limit",
    str(MAX_MODEL_LEN),
    "--attempts",
    str(SOLVER_ATTEMPTS),
    "--distractors",
    str(SOLVER_DISTRACTORS),
]
if SAMPLE_LIMIT:
    filter_cmd += ["--limit", str(SAMPLE_LIMIT)]
filter_started = time.monotonic()
subprocess.run(filter_cmd, check=True)
print(f"filtered in {(time.monotonic() - filter_started) / 60:.1f} min")

In [ ]:
# X, read two ways, because they answer different questions.
def read_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


solved = read_rows(SOLVED)
rejected = read_rows(REJECTED)
judged = solved + rejected
assert judged, "Nothing was judged."
tries = sum(int(row.get("of_attempts", SOLVER_ATTEMPTS)) for row in judged)
hits = sum(int(row.get("solved_attempts", 0)) for row in judged)
# pass@1 is the like-for-like comparison with production, which gets one try per attempt and
# stops at the first success. pass@k is what the filter keeps, and it is the higher number.
pass_at_1 = hits / tries if tries else 0.0
pass_at_k = len(solved) / len(judged)
print(f"judged {len(judged)}   kept {len(solved)}")
print(f"X (pass@1)   {pass_at_1:.4f}")
print(f"X (pass@{SOLVER_ATTEMPTS})   {pass_at_k:.4f}")
if not SOLVER_DISTRACTORS:
    print("Both are measured with gold tables only, so both are upper bounds on production X.")
print("\nby family, pass@1:")
for family in families:
    rows = [row for row in judged if str(row.get("family")) == family]
    if not rows:
        continue
    family_tries = sum(int(row.get("of_attempts", SOLVER_ATTEMPTS)) for row in rows)
    family_hits = sum(int(row.get("solved_attempts", 0)) for row in rows)
    print(f"  {family:12s} {family_hits / family_tries:.4f}  ({len(rows)} questions)")

SUMMARY = ARTIFACTS / f"{SPLIT}_x_measurement.json"
SUMMARY.write_text(
    json.dumps(
        {
            "project_revision": PROJECT_SHA,
            "model": MODEL,
            "model_revision": MODEL_REVISION,
            "split": SPLIT,
            "programs_sha256": sha256(PROGRAMS),
            "attempts": SOLVER_ATTEMPTS,
            "distractors": SOLVER_DISTRACTORS,
            "rendered": len(read_rows(RENDERED)),
            "judged": len(judged),
            "kept": len(solved),
            "x_pass_at_1": pass_at_1,
            f"x_pass_at_{SOLVER_ATTEMPTS}": pass_at_k,
            "by_family": {
                family: {
                    "questions": len([r for r in judged if str(r.get("family")) == family]),
                    "hits": sum(
                        int(r.get("solved_attempts", 0))
                        for r in judged
                        if str(r.get("family")) == family
                    ),
                }
                for family in families
            },
        },
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print("\nwrote", SUMMARY)
print("Download artifacts/ and keep it: outputs/ is outside git, so this is the only copy.")